# 2. Service-to-Service with Client Credentials

**Scenario**: a background daemon (cron job, worker, scheduled function) needs to call an API. There's no user sitting in front of a browser. This is the canonical *service-to-service* (S2S) pattern.

The flow is called **client credentials**:

```
daemon ──(client_id + client_secret)──▶ Entra ID  ──▶ access token
daemon ──(Bearer token)─────────────────▶ api-b    ──▶ data
```

Entra treats the daemon like any other principal. It checks:
1. Does the secret match?
2. Has the daemon's service principal been **granted** the app-role it's asking for, on the target API?

The token it issues carries the granted roles in the `roles` claim.

## The `/.default` scope

When you do client credentials, you can't pick individual scopes — you ask for `<resource>/.default`, meaning *"give me everything this app has been granted on this resource"*. That's an Entra-ism: delegated scopes need user consent, so for the app-only case the `/.default` trick hands back all pre-consented app-roles.

In [1]:
import httpx, json, base64

TOKEN_URL = 'http://localhost:9100/contoso/oauth2/v2.0/token'
API_B     = 'http://localhost:8002'

def decode_payload(t):
    p = t.split('.')[1]
    return json.loads(base64.urlsafe_b64decode(p + '=' * (-len(p) % 4)))

# --- 1. Get a token as the daemon app ---
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'client_credentials',
    'client_id': 'daemon-client-id',
    'client_secret': 'daemon-secret-value',
    'scope': 'api://api-b/.default',
})
r.raise_for_status()
token = r.json()['access_token']
print(json.dumps(decode_payload(token), indent=2))

{
  "iss": "http://localhost:9100/contoso/v2.0",
  "iat": 1776711725,
  "nbf": 1776711725,
  "exp": 1776715325,
  "jti": "128c05f6-1bc5-4345-a732-950f593c4f67",
  "aud": "api://api-b",
  "azp": "daemon-client-id",
  "sub": "daemon-client-id",
  "oid": "sp-daemon-client-id",
  "roles": [
    "Files.Read.All"
  ],
  "tid": "contoso"
}


Notice:
- `aud = api://api-b` — token is *for* API-B, not usable elsewhere.
- `roles = ['Files.Read.All']` — the app role granted to the daemon.
- **No `upn` / `scp`** — this is an app-only token; no user was involved.

## 2. Call API-B

In [2]:
r = httpx.get(f'{API_B}/files', headers={'Authorization': f'Bearer {token}'})
print(r.status_code)
print(json.dumps(r.json(), indent=2))

200
{
  "mode": "app-only",
  "caller": "daemon-client-id",
  "files": [
    {
      "id": "f1",
      "owner": "alice@contoso.com",
      "name": "design-doc.md"
    },
    {
      "id": "f2",
      "owner": "alice@contoso.com",
      "name": "budget.xlsx"
    },
    {
      "id": "f3",
      "owner": "bob@contoso.com",
      "name": "plan.pdf"
    }
  ]
}


## 3. What happens if the app role was **not granted**?

A `client_credentials` token succeeds as long as the secret is valid — but the token only carries the roles the **service principal was granted** on the target resource. If the role is missing, you still get a token, but the target API will reject the call with **403**.

We seeded a second daemon, `reporting-daemon`, that has **no** granted roles on `api-b`. Let's see what happens.


In [3]:
# Token issues fine (secret is valid), but the `roles` claim will be empty.
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'client_credentials',
    'client_id': 'reporting-daemon-client-id',
    'client_secret': 'reporting-daemon-secret-value',
    'scope': 'api://api-b/.default',
})
no_role_token = r.json()['access_token']
print('roles claim:', decode_payload(no_role_token).get('roles'))

# API-B rejects: no Files.Read.All role.
r2 = httpx.get(f'{API_B}/files', headers={'Authorization': f'Bearer {no_role_token}'})
print('status:', r2.status_code, r2.json())

# Bonus: what about a wrong secret? Entra refuses at the token endpoint itself.
r3 = httpx.post(TOKEN_URL, data={
    'grant_type': 'client_credentials',
    'client_id': 'daemon-client-id',
    'client_secret': 'WRONG',
    'scope': 'api://api-b/.default',
})
print('bad secret ->', r3.status_code, r3.json())


roles claim: []
status: 403 {'detail': 'missing app role: Files.Read.All'}
bad secret -> 401 {'detail': 'invalid_client'}


## 4. What happens with a token for the wrong API?

Tokens include `aud` (audience). API-B rejects anything that isn't `api://api-b`.

In [4]:
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'client_credentials',
    'client_id': 'daemon-client-id',
    'client_secret': 'daemon-secret-value',
    'scope': 'api://api-a/.default',   # wrong target
})
other = r.json().get('access_token')
r2 = httpx.get(f'{API_B}/files', headers={'Authorization': f'Bearer {other}'})
print('status:', r2.status_code, r2.json())

status: 401 {'detail': 'token validation failed: Invalid audience'}


That's the whole point of `aud` — a stolen token from one API can't be replayed against another.

## 5. Using MSAL (what you'd actually write in production)

Nobody POSTs to the token endpoint by hand in real code — you use Microsoft's **MSAL** library. It handles caching, refresh, retries, certificate auth, etc.

**MSAL snippet (for real Entra, not runnable against our mock — MSAL rejects non-HTTPS authorities):**

```python
from msal import ConfidentialClientApplication

app = ConfidentialClientApplication(
    client_id='<AZURE_CLIENT_ID>',
    client_credential='<AZURE_CLIENT_SECRET>',
    authority='https://login.microsoftonline.com/<tenant-id>',
)
result = app.acquire_token_for_client(scopes=['api://api-b/.default'])
token = result['access_token']   # cached + auto-refreshed
```

MSAL adds caching, automatic refresh, certificate auth, and confidential-client flows on top of the raw HTTP calls we used above.


In real Azure you'd use:
```python
ConfidentialClientApplication(
    client_id=os.environ['AZURE_CLIENT_ID'],
    client_credential=os.environ['AZURE_CLIENT_SECRET'],
    authority=f'https://login.microsoftonline.com/{tenant_id}',
)
```

Even better — don't use a secret at all. **Managed identity** (notebook 3) removes the need entirely.

## Summary

- **Client credentials** = app authenticates with its own creds, gets app-only token.
- Token carries `roles`, *not* `scp` — these are application permissions.
- Always call `<resource>/.default` for client credentials.
- Audience (`aud`) scoping stops token replay across APIs.
- Use MSAL, not raw HTTP, in real code.